In [ ]:
# ============================================
# rag56_first_run_and_doc_hint_fixes.ipynb
#
# [실험 목적]
# rag-56 골든셋을 처음으로 전체 돌려보면서, 그동안
# core40으로만 검증하던 파이프라인이 rag-56에서는 어떤 새로운 오탐·
# 누락 패턴을 보이는지 확인하고, GKL·한국철도공사 관련 문서 힌트 매칭
# 오류를 잡아나가는 게 목적
#
# [진행 방식과 알아낸 것]
#
# 1. rag-56 최초 전체 채점 (cell 11~14)
#    - ask_rfp_final()로 56문항 답변을 생성해 official_score()로 채점
#    - 0점 나온 6개 문항(c03, c09, c18, c23, g18, g23)을 추려서 개별 원인 조사 시작
#
# 2. GKL("그랜드코리아레저") 문서 힌트 매칭 실패 원인 조사 (cell 15~25)
#    - "GKL이 임직원 협업과 전자결재 환경을..." 질문에서 문서 힌트가
#      아예 안 잡히는 문제 발견
#    - ORG_ALIAS_MAP에 "그랜드코리아레저(주)"를 등록했는데도 안 잡혀서
#      원인을 단계별로 추적:
#      1) "2024년"이라는 흔한 연도 표현이 COMMON_SUFFIX_WORDS에 없어서
#         노이즈로 작용 -> 추가
#      2) COMMON_SUFFIX_WORDS에 추가해도 실제 매칭 단계는
#         COMMON_FILENAME_WORDS를 참조하고 있어서 효과 없음 -> 이쪽에도 추가
#      3) 그래도 안 잡혀서 ORG_ALIAS_MAP 키 자체를 확인해보니, 파일명에서
#         기관명을 추출할 때 괄호("(주)")를 이미 제거하는데 별칭 딕셔너리
#         키는 "그랜드코리아레저(주)"로 괄호가 남아있어 매칭 자체가 안
#         되고 있었음(딕셔너리 키와 실제 추출된 기관명 형태가 불일치)
#      4) 키를 "그랜드코리아레저"(괄호 없이)로 수정해서 최종 해결
#    - 하나의 매칭 실패가 실제로는 "노이즈 단어 미등록 -> 잘못된 목록에
#      등록 -> 키 형태 불일치"까지 세 가지 원인이 겹쳐 있었다는 걸
#      순차적으로 밝혀낸 셀
#
# 3. 한국철도공사 문서 힌트 매칭(운행정보기록) 재조사 (cell 28~34)
#    - "한국철도공사가 열차 운행기록을 자동으로 분석하는..." 질문이
#      같은 발주기관의 다른 문서(예약발매시스템, 모바일오피스)와
#      헷갈리는 문제를 디버그 함수(_debug_fuzzy~fuzzy3)로 min_overlap과
#      조사 제거 방식을 바꿔가며 단계적으로 원인 추적
#      (이 조사가 이후 generation_prompt_and_variance_experiment의
#      extract_doc_hints_multi_v2 개선으로 이어짐)
#
# 4. 수정 반영 후 rag-56 전체 재채점 (cell 36~41)
#    - GKL 별칭 수정을 반영한 최종 버전으로 56문항 재생성·재채점
#    - c11, g08, g11, g21 등 이후에도 계속 재조사되는 문항들을 이 시점에
#      먼저 개별 확인
# ============================================

In [1]:
# 드라이브 마운트

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 청크 데이터 로드

import sys, types
import pickle, re, json
import numpy as np
from pathlib import Path

src_module = types.ModuleType('src')
chunking_module = types.ModuleType('src.chunking')
class Chunk:
    pass
chunking_module.Chunk = Chunk
src_module.chunking = chunking_module
sys.modules['src'] = src_module
sys.modules['src.chunking'] = chunking_module

DATA_DIR = Path('/content/drive/MyDrive/중급 프로젝트')

with open(DATA_DIR / 'chunks.pkl', 'rb') as f:
    chunk_objects = pickle.load(f)

all_chunks_final = []
chunk_metadata_final = []
for c in chunk_objects:
    all_chunks_final.append(c.text)
    meta_info = c.metadata if c.metadata else {}
    chunk_metadata_final.append({
        '파일명': c.doc_id,
        '발주기관': meta_info.get('발주_기관', ''),
        '사업금액': meta_info.get('사업_금액', None),
        '마감일': meta_info.get('입찰_참여_마감일', ''),
    })
print(f"청크: {len(all_chunks_final)}개")

청크: 18239개


In [4]:
# KURE 임베딩 인덱스 로드

import torch
import faiss
from sentence_transformers import SentenceTransformer

with open(DATA_DIR / 'kure_embeddings.pkl', 'rb') as f:
    kure_embeddings = pickle.load(f)

index_kure = faiss.IndexFlatL2(kure_embeddings.shape[1])
index_kure.add(np.array(kure_embeddings).astype('float32'))

kure_model = SentenceTransformer('nlpai-lab/KURE-v1', device='cuda', model_kwargs={'torch_dtype': torch.float16})
print(kure_model.device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/807 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

cuda:0


In [5]:
# OpenAI 클라이언트 + 고유 문서 목록 추출

from google.colab import userdata
import openai

api_key = userdata.get('OPENAI_API_KEY')
client = openai.OpenAI(api_key=api_key)

seen = set()
all_filenames_with_biz = []
for cm in chunk_metadata_final:
    if cm['파일명'] not in seen:
        seen.add(cm['파일명'])
        all_filenames_with_biz.append((cm['파일명'], cm.get('발주기관', '')))
print(f"고유 문서 수: {len(all_filenames_with_biz)}")

고유 문서 수: 98


In [6]:
# 기관명 별칭 / 공통 단어 블랙리스트 / 법률 키워드 맵 / 표 보정 데이터

ORG_ALIAS_MAP = {
    '대검찰청': ['검찰'],
    '고려대학교': ['고려대'],
    '한국산업단지공단': ['산단'],
}

COMMON_SUFFIX_WORDS = {
    '박물관', '시스템', '센터', '공단', '진흥원', '협회', '재단', '연구원', '공사', '대학교',
    '사업', '관리', '운영', '구축', '개선', '개발', '지원', '정보', '용역', '기관', '기술',
    '고도화', '확대', '기능', '서비스', '일자리', '플랫폼', '통합', '접수',
    '일자리재단', '일자리플랫폼', '보험', '입찰공고', '공고',
    '과학연구', '과학연', '학연구', '연구소', '기록관리', '경기기록',
    '학교', '학교 ', ' 학교', '산학협력단', '산학협력', '학협력단',
    '통합시스템'
}
COMMON_FILENAME_WORDS = COMMON_SUFFIX_WORDS | {'용역', '수립', '2차', '1차', '3차', '운영', '및', '구축용역'}

LEGAL_KEYWORDS_MAP = {
    '하도급': ['하도급'],
    '공동수급': ['공동수급', '지분율', '컨소시엄'],
    '지분율': ['지분율', '공동수급'],
    '계약보증금': ['계약보증금', '보증금'],
    '평가': ['배점', '평가비율', '기술평가', '가격평가'],
    '제안서 보상': ['제안서 보상'],
    '불이익': ['부정당업자', '입찰보증금', '귀속'],
    '제출물': ['제출서류', '부', 'USB', '제출규격'],
    '제출': ['제출서류', 'USB'],
    '수량': ['부', 'USB'],
    '구축기간': ['사업기간', '구축기간', '개월'],
    '사업기간': ['사업기간', '구축기간', '개월'],
    '유지보수': ['무상유지보수', '유지보수기간', '하자보수', '무상 하자보수'],
    '참가자격': ['참가자격', '참가 자격'],
    '유지관리': ['하자보수', '유지관리 인력', '무상 하자보수'],
    '교육 의무': ['유지관리 인력', '사용자 및 관리자', '하자보수'],
    '교육을': ['유지관리 인력', '사용자 및 관리자', '하자보수'],
    '검수 후': ['하자보수', '유지관리 인력'],
    '재입찰': ['재입찰', '재공고입찰', '최초의 입찰'],
    '재공고': ['재입찰', '재공고입찰', '최초의 입찰'],
    '조건 변경': ['재입찰', '재공고입찰', '최초의 입찰'],
    '지역 요건': ['주된 영업소', '소재지'],
    '부산에': ['주된 영업소', '소재지'],
    '지역요건': ['주된 영업소', '소재지'],
    '소재지': ['주된 영업소', '소재지'],
    '보유인력': ['보유인력', '배점한도'],
    '배점한도': ['보유인력', '배점한도'],
    '계량평가': ['보유인력', '배점한도', '재무구조'],
    '규모비율': ['규모비율', '환산점수', '점수비중'],
    '환산점수': ['규모비율', '환산점수', '점수비중'],
    '수행실적': ['규모비율', '환산점수', '수행실적'],
    '신인도': ['신인도', '가점'],
    '가점표': ['신인도', '가점'],
    '연구원 승인': ['Lesson', '회람'],
    '발생한 경우': ['Lesson', '회람'],
    '회람': ['Lesson', '회람'],
}

In [7]:
# 조건 필터링 + 검색 함수

def find_relevant_keywords(question):
    matched = []
    for trigger, kws in LEGAL_KEYWORDS_MAP.items():
        if trigger in question:
            matched.extend(kws)
    return list(set(matched))

def is_aggregation_question(question):
    keywords = ['몇 개', '개수', '다 나열', '몇 건']
    strong_total = '전부' in question or ('총' in question and ('개' in question or '건' in question))
    return any(kw in question for kw in keywords) or strong_total

def extract_filter_conditions(query):
    conditions = {}
    if '억' in query and ('이상' in query or '넘는' in query):
        match = re.search(r'(\d+)억', query)
        if match:
            conditions['금액_최소'] = int(match.group(1)) * 100000000
    if '지자체' in query or '지방자치단체' in query:
        conditions['지자체'] = True
    if '공사' in query and ('OO공사' in query or '발주기관이' in query):
        conditions['공사'] = True
    if 'AI' in query:
        conditions['주제_AI'] = True
    if '긴급' in query:
        conditions['긴급'] = True
    if '보안' in query:
        conditions['보안'] = True
    if '재난' in query:
        conditions['재난'] = True
    return conditions

def is_local_gov(org):
    if org is None or (isinstance(org, float)):
        return False
    return bool(re.search(r'(광역시|특별시|특별자치도|특별자치시|[가-힣]+도|[가-힣]+시|[가-힣]+군|[가-힣]+구)$', str(org).strip()))

def get_filtered_candidates(conditions, chunk_metadata):
    if not conditions:
        return None
    doc_info = {}
    for cm in chunk_metadata:
        fname = cm['파일명']
        if fname not in doc_info:
            doc_info[fname] = cm
    allowed = set()
    for fname, info in doc_info.items():
        ok = True
        if '금액_최소' in conditions:
            amt = info.get('사업금액')
            if amt is None or amt < conditions['금액_최소']:
                ok = False
        if conditions.get('지자체'):
            if not is_local_gov(info.get('발주기관')):
                ok = False
        if conditions.get('공사'):
            org = str(info.get('발주기관', ''))
            if '공사' not in org:
                ok = False
        if conditions.get('긴급'):
            if '긴급' not in fname:
                ok = False
        if conditions.get('재난'):
            if '재난' not in fname:
                ok = False
        if ok:
            allowed.add(fname)
    return allowed if allowed else None

def search_with_filter(query, index, model, chunk_metadata, all_chunks, k=10, max_per_doc=1):
    conditions = extract_filter_conditions(query)
    allowed_filenames = get_filtered_candidates(conditions, chunk_metadata)
    query_embedding = model.encode([query])
    search_k = min(len(all_chunks), 2000)
    distances, indices = index.search(np.array(query_embedding).astype('float32'), search_k)
    seen_docs = {}
    results = []
    for i in indices[0]:
        doc_name = chunk_metadata[i]['파일명']
        if allowed_filenames is not None and doc_name not in allowed_filenames:
            continue
        count = seen_docs.get(doc_name, 0)
        if count < max_per_doc:
            results.append(i)
            seen_docs[doc_name] = count + 1
        if len(results) >= k:
            break
    return results

def normalize_org_name(name):
    return re.sub(r'(특별시|광역시|특별자치시|특별자치도)', '', name)

In [8]:
# 8. 문서 힌트 추출 (핵심 로직, 최종본)

def extract_doc_hints_multi(question, all_filenames_with_biz):
    q_no_space = question.replace(' ', '').replace('&', '')
    org_candidates = []
    for fname, biz_name in all_filenames_with_biz:
        org_part = fname.replace('refined_', '').split('_')[0].strip()
        org_core = re.sub(r'\s*\(.*?\)\s*', '', org_part).strip()
        org_core_clean = re.sub(r'^\(사\)', '', org_core).strip()
        org_core_clean = re.sub(r'\s*입찰공고\s*$', '', org_core_clean).strip()
        org_core_norm = normalize_org_name(org_core_clean)
        if len(org_core_clean) < 2:
            continue

        matched = False
        if org_core_clean in question:
            matched = True
        elif len(org_core_norm) >= 3 and org_core_norm in question:
            matched = True
        elif org_core_clean in ORG_ALIAS_MAP and any(alias in question for alias in ORG_ALIAS_MAP[org_core_clean]):
            matched = True
        else:
            min_len = 4
            for target_str in [org_core_clean, org_core_norm]:
                for start in range(len(target_str) - min_len + 1):
                    for length in range(len(target_str) - start, min_len - 1, -1):
                        substr = target_str[start:start+length]
                        if substr.strip() in question and substr.strip() not in COMMON_SUFFIX_WORDS:
                            matched = True
                            break
                    if matched:
                        break
                if matched:
                    break
        if matched:
            org_candidates.append((fname, org_core_clean))

    biz_candidates = []
    quoted = re.findall(r"['\"]([^'\"]+)['\"]", question)
    for fname, biz_name in all_filenames_with_biz:
        biz_name = str(biz_name).strip()
        if len(biz_name) >= 4 and biz_name in question:
            biz_candidates.append(fname)
            continue
        for q in quoted:
            if q in biz_name or biz_name in q:
                biz_candidates.append(fname)
                break
        eng_words = re.findall(r'[A-Za-z][A-Za-z&\s]{2,}[A-Za-z]', biz_name)
        for ew in eng_words:
            ew_no_space = ew.strip().replace(' ', '').replace('&', '')
            if len(ew_no_space) >= 4 and ew_no_space in q_no_space:
                biz_candidates.append(fname)
                break

    stopwords_general = {'사업의', '사업에서', '사업은', '어떻게', '되나요', '되나요?', '몇', '어떤', '얼마', '비교', '알려줘', '정리해줘', '무엇인가요', '관련', '입찰공고일', '공고일', '입찰공고'}
    raw_keywords = [w.rstrip('.,?!') for w in re.split(r'[ ,·]', question) if len(w) >= 4]
    keywords_all = [w for w in raw_keywords if w not in stopwords_general and w not in COMMON_FILENAME_WORDS and '입찰공고' not in w]

    def fuzzy_match(kw, text, min_overlap=4):
        kw_ns = kw.replace(' ', '')
        text_ns = text.replace(' ', '')
        if kw_ns in text_ns:
            return True
        for n in range(len(kw_ns), min_overlap - 1, -1):
            if kw_ns[:n] in text_ns:
                return True
        return False

    def keyword_weight(kw):
        return 3 if re.search(r'[A-Za-z]', kw) else 1

    filename_candidates = []
    for fname, biz_name in all_filenames_with_biz:
        fname_clean = fname.replace('refined_', '').replace('.hwp', '').replace('.pdf', '')
        matched_kws = [kw for kw in keywords_all if fuzzy_match(kw, fname_clean)]
        score = sum(keyword_weight(kw) for kw in matched_kws)
        if score > 0:
            filename_candidates.append((fname, score, len(matched_kws)))

    if filename_candidates:
        filename_candidates.sort(key=lambda x: -x[1])
        max_score = filename_candidates[0][1]
        for top_fname, score, cnt in filename_candidates:
            if score >= max_score * 0.6 or score >= 1:
                if top_fname not in [f for f, _ in org_candidates] and top_fname not in biz_candidates:
                    if len(filename_candidates) <= 3 or score >= max(max_score * 0.6, 1):
                        biz_candidates.append(top_fname)

    org_groups = {}
    for fname, org_core in org_candidates:
        org_groups.setdefault(org_core, []).append(fname)

    stopwords = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교'}
    keywords = [w for w in re.split(r'[ ,]', question) if len(w) >= 2 and w not in stopwords]

    final_hints = []
    for org_core, fnames in org_groups.items():
        fnames = list(set(fnames))
        if len(fnames) == 1:
            final_hints.append(fnames[0])
        else:
            fname_to_biz = dict(all_filenames_with_biz)
            best_doc, best_score2 = None, -1
            for fname in fnames:
                biz_name = fname_to_biz.get(fname, '')
                score2 = sum(1 for kw in keywords if kw in fname or kw in str(biz_name))
                if score2 > best_score2:
                    best_score2, best_doc = score2, fname
            final_hints.append(best_doc)

    for fname in biz_candidates:
        if fname not in final_hints:
            final_hints.append(fname)

    return list(dict.fromkeys(final_hints))

In [9]:
# 프롬프트

SYSTEM_PROMPT_NEW_V2 = """
너는 'RFP 챗봇'이야. 입찰메이트 컨설턴트가 제안요청서(RFP) 문서를 빠르게 파악할 수 있게 도와줘.

## 기본 원칙

1. 반드시 아래에 제공된 문서 내용(컨텍스트)에 근거해서만 답변해. 문서에 없는 내용을 추측하거나 지어내지 마.

2. 답변은 간결하고 명확하게 작성해. 불필요한 서론 없이 핵심부터 답해.

3. 질문 유형에 따라 답변 형식을 다르게 해:
   - 단일 사실 조회 (예: "예산이 얼마야?") → 핵심 수치/사실 위주로 간결하게
   - 두 개 이상 비교 (예: "A랑 B 중 뭐가 더 커?") → 각 항목을 나란히 제시하고 비교 결론 제시
   - 목적/배경을 묻는 질문 → 관련 섹션을 요약해서 설명
   - 조건에 맞는 여러 문서를 찾는 질문 → 목록 형태로 정리

4. 이전 대화에서 언급된 문서나 주제가 있으면, 후속 질문("그럼 마감일은?" 등)은 같은 문서/주제 맥락에서 답변해.

5. 답변 끝에는 근거가 된 문서명을 명시해.

## 답변을 거절/기권해야 하는 경우 (매우 중요)

아래 경우에는 문서 안에서 관련 정보를 억지로 찾아서 답하려 하지 말고, 명확히 "답변할 수 없다"고만 말하고 끝내. 관련 있어 보이는 부가 정보를 나열하지 마.

- **범위 밖 요청(out_of_scope)**: 네가 할 수 없는 행동을 요청하는 경우(전화 걸기, 이메일 보내기, 실시간 조회 등), 또는 "오늘", "지금", "최신"처럼 실시간·최신 정보를 요구하는 경우. 이때는 "이 기능은 제가 수행할 수 없습니다" 또는 "실시간 정보는 제공된 문서에서 확인할 수 없습니다"라고만 답하고, 대신 관련 문서를 찾아주거나 연락처를 나열하는 등 다른 시도를 하지 마.

- **근거 부족(insufficient_evidence)**: 낙찰 결과, 경쟁사 현황, 예상 낙찰가처럼 애초에 이 문서(제안요청서)에 있을 수 없는 정보를 물어보는 경우. "확인되지 않습니다"라고만 답해.

- **판단/추측 요청(ambiguous)**: "우리 회사가 자격을 충족하는지 판정해줘", "수주 확률이 얼마냐" 처럼 사용자의 상황과 문서를 대조해서 네가 주관적으로 판단·확률을 계산해야 하는 질문. 이런 판정이나 확률 계산은 네가 할 수 없다고 답하고, 판단에 필요한 조건 목록만 간단히 안내해도 되지만 장황하게 체크리스트를 만들지는 마.

- **사용자가 임의의 가정을 세우고 그 가정으로 확정 답변을 요구하는 경우**: "문서에 없으면 OO라고 가정하고 확정해줘"처럼, 사용자가 제시한 임의의 규칙(추측)을 근거 삼아 사실인 것처럼 답을 만들어달라는 요청. 이건 절대 받아들이지 마. "문서에 없는 정보는 임의로 가정해서 확정할 수 없습니다"라고 답하고, 사용자가 제안한 가정을 그대로 적용해서 계산해주지 마.

## 표 형식 데이터 안내

컨텍스트에 [표]라는 표시와 함께 "항목 | 값" 형태로 된 부분이 나오면, 이는 원본 문서의 표를 옮긴 것이야. 각 줄은 표의 한 행을 의미하고, |로 구분된 각 항목은 표의 열(칸)을 의미해. 이 형식을 참고해서 항목과 값을 정확히 짝지어 답변해.

## 여러 문서 처리

컨텍스트에 여러 문서의 내용이 섞여 있을 수 있어. 각 문서 조각이 어느 문서(파일명)에서 왔는지 구분해서, 서로 다른 문서의 정보를 혼동하거나 섞어서 답하지 마.

일부 정보(예: 긴급 여부, 재공고 여부)는 본문 내용이 아니라 문서명(파일명)에만 표시되어 있을 수 있어. 문서명에 이런 정보가 있으면 그것도 근거로 활용해서 답해.

## 주제/카테고리 판단 시 주의사항

질문의 키워드와 문서 안의 유사한 단어가 겉보기에 비슷해 보여도, 실제 의미는 다를 수 있어. 문서의 실제 사업 목적과 내용까지 확인해서 질문 의도와 정확히 일치하는지 판단하고, 확신이 안 서면 "이 문서는 [실제 의미]를 다루고 있어 질문 의도와 다를 수 있습니다"처럼 구분해서 답해. 단어의 표면적 유사성만으로 포함시키지 마.

아래는 실제로 혼동이 발생했던 사례야. 반드시 참고해서 판단해:

예시: "재난 관련 사업을 찾아줘"라는 질문에, 사업명이 "적십자병원 병원정보 재해복구시스템 구축 용역"인 문서가 검색됐다고 하자. 이 문서는 재난(자연재해, 재난관리) 관련 사업이 아니야. "재해복구시스템(Disaster Recovery System)"은 서버/데이터베이스 장애 시 데이터를 복구하는 IT 인프라 용어이고, "병원정보시스템 데이터베이스 운영"을 다루는 순수 IT 시스템 구축 사업이야. 이 사업을 "재난 관련"으로 포함시키면 틀린 답변이야. 반드시 제외해.

마찬가지로 "응급의료 상황관리시스템"(병원 전원·환자 이송을 지원하는 IT 시스템)도 "재난 관리 시스템"과는 다른 목적의 사업이야. 재난은 지진, 홍수, 화재 등 자연재해나 사회재난에 대응하는 시스템을 뜻하며, 단순히 "응급", "긴급", "재해" 같은 단어가 사업명에 있다고 해서 재난 관련 사업으로 분류하면 안 돼.

## 질문 해석 관련

질문에 "OO", "XX" 같은 placeholder처럼 보이는 표현이 있어도, 이는 실제로 채워야 할 빈칸이 아니라 "특정 패턴을 가진 이름 전체"를 가리키는 일반적인 화법일 수 있어. 예를 들어 "발주기관이 OO공사인 사업"은 "발주기관명이 '공사'로 끝나는 모든 사업"을 뜻하는 것이지, 사용자가 실제 공사명을 지정해줘야 한다는 뜻이 아니야. 이런 경우 되묻지 말고, 컨텍스트 안에서 해당 패턴에 맞는 사업을 최대한 찾아서 답해.

## 금액 표기 관련

금액은 부가세(VAT) 포함/별도 표기가 문서마다 다를 수 있어. 답변할 때 원문에 표기된 형태(포함/별도 여부 포함) 그대로 전달하고, 임의로 환산하지 마.

## 구조화된 필드(공고번호, 사업금액, 입찰 참여 시작일/마감일, 발주기관) 답변 규칙

이 필드들은 컨설턴트의 실제 입찰 결정에 직결되니까 특히 신중하게 답해.

- 검색된 문서 조각과 메타데이터에 명확한 값이 있으면, 근거와 함께 답변해.
- 값이 없거나 불확실하면 절대 추정하지 말고 "확인되지 않습니다"라고 명확히 답해.
- 아래 함정에 특히 주의해:
  - 공고번호를 유사한 다른 번호나 제목의 "[재공고]" 표시만으로 추정하지 마.
  - 개찰 시각이나 제안서 평가 시각을 입찰 참여 마감일로 착각해서 답하지 마. 이 셋은 서로 다른 시점이야.
  - 공개일(공고가 게시된 날짜)을 입찰 참여 시작일로 대체하지 마.
  - 발주기관은 게시기관·수요기관·계약기관이 다를 수 있으니까, 근거 없이 하나를 임의로 선택하지 마.
  - 사업금액이 0원이나 1원으로 보이면, 이건 실제 금액이 아니라 비공개·미확정을 나타내는 표시일 수 있어. 이 경우 실금액처럼 답하지 말고 "금액이 비공개이거나 미확정 상태로 보입니다"라고 답해.

## 참가자격 / 제한조건 / 평가기준 / 제출요건 / 계약 리스크(위약금, 계약보증금 등) 답변 규칙

이 항목들도 컨설턴트가 실제로 입찰 여부를 판단하고 계약 의무를 이해하는 데 직결되니까 신중하게 답해.

- 검색된 문서 조각 안에 명확한 근거가 있을 때만 답변해.
- 명확한 근거가 없으면 "제공된 문서 범위에서는 확인되지 않습니다. 원문 전체 확인이 필요할 수 있습니다"라고 답해.
- 다른 사업의 일반적인 조항이나 통상적인 관행을 이 사업에 적용해서 답하지 마.

## 부분 정보 처리

질문에 여러 정보가 섞여 있고 그중 일부만 확인 가능하면, 확인되는 정보는 근거와 함께 답하고 확인 안 되는 정보만 위 규칙에 따라 "확인되지 않습니다"라고 답해. 일부가 확인 안 된다고 전체 답변을 포기하지 마.

## 컨텍스트 (검색된 문서 조각)
{context}

## 질문
{question}
"""

In [10]:
# 답변 생성 함수

def ask_rfp_final(question, model_name="gpt-5-mini", max_retries=2):
    doc_hints = extract_doc_hints_multi(question, all_filenames_with_biz)
    doc_hints = doc_hints[:3]
    keywords = find_relevant_keywords(question)
    conditions = extract_filter_conditions(question)
    allowed = get_filtered_candidates(conditions, chunk_metadata_final)

    fname_to_meta = {}
    for cm in chunk_metadata_final:
        if cm['파일명'] not in fname_to_meta:
            fname_to_meta[cm['파일명']] = cm

    def meta_header(fname):
        m = fname_to_meta.get(fname, {})
        org = m.get('발주기관', '')
        amt = m.get('사업금액')
        amt_str = f"{amt:,.0f}원" if amt not in (None, '') else "확인되지 않음"
        return f"[문서: {fname}]\n[발주기관(메타데이터): {org}]\n[사업금액(메타데이터): {amt_str}]"

    context_parts = []

    if is_aggregation_question(question) and len(doc_hints) >= 1:
        stopwords_q = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교'}
        qkeywords = [w for w in re.split(r'[ ,]', question) if len(w) >= 2 and w not in stopwords_q]
        best_doc, best_score = doc_hints[0], -1
        for fname in doc_hints:
            biz = fname_to_meta.get(fname, {}).get('발주기관', '')
            score = sum(1 for kw in qkeywords if kw in fname or kw in str(biz))
            if score > best_score:
                best_score, best_doc = score, fname
        doc_hint = best_doc
        doc_chunks = [i for i, cm in enumerate(chunk_metadata_final) if cm['파일명'] == doc_hint]
        header = meta_header(doc_hint)
        for i in doc_chunks:
            context_parts.append(f"{header}\n{all_chunks_final[i]}")

    elif len(doc_hints) == 1 and keywords:
        doc_hint = doc_hints[0]
        doc_chunks = [i for i, cm in enumerate(chunk_metadata_final) if cm['파일명'] == doc_hint]
        keyword_chunks = [i for i in doc_chunks if any(kw in all_chunks_final[i] for kw in keywords)]
        result_indices = keyword_chunks[:25] if keyword_chunks else search_with_filter(question, index_kure, kure_model, chunk_metadata_final, all_chunks_final, k=10, max_per_doc=5)
        for i in result_indices:
            fname = chunk_metadata_final[i]['파일명']
            context_parts.append(f"{meta_header(fname)}\n{all_chunks_final[i]}")

    elif len(doc_hints) >= 2:
        for doc_hint in doc_hints:
            doc_chunks = [i for i, cm in enumerate(chunk_metadata_final) if cm['파일명'] == doc_hint]
            if keywords:
                matched = [i for i in doc_chunks if any(kw in all_chunks_final[i] for kw in keywords)]
                selected = matched[:8] if matched else doc_chunks[:8]
            else:
                selected = doc_chunks[:8]
            header = meta_header(doc_hint)
            for i in selected:
                context_parts.append(f"{header}\n{all_chunks_final[i]}")
    elif doc_hints:
        doc_hint = doc_hints[0]
        doc_chunks = [i for i, cm in enumerate(chunk_metadata_final) if cm['파일명'] == doc_hint]
        header = meta_header(doc_hint)
        for i in doc_chunks[:15]:
            context_parts.append(f"{header}\n{all_chunks_final[i]}")
    elif allowed:
        k = min(len(allowed), 80)
        result_indices = search_with_filter(question, index_kure, kure_model, chunk_metadata_final, all_chunks_final, k=k, max_per_doc=1)
        for i in result_indices:
            fname = chunk_metadata_final[i]['파일명']
            context_parts.append(f"{meta_header(fname)}\n{all_chunks_final[i]}")
    else:
        result_indices = search_with_filter(question, index_kure, kure_model, chunk_metadata_final, all_chunks_final, k=10, max_per_doc=3)
        for i in result_indices:
            fname = chunk_metadata_final[i]['파일명']
            context_parts.append(f"{meta_header(fname)}\n{all_chunks_final[i]}")

    context = "\n\n---\n\n".join(context_parts)
    final_prompt = SYSTEM_PROMPT_NEW_V2.format(context=context, question=question)

    for attempt in range(max_retries):
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": final_prompt}],
            max_completion_tokens=8000,
            reasoning_effort="low"
        )
        answer = response.choices[0].message.content
        if answer:
            return answer
    return "(답변 생성 실패)"

In [11]:
# 통합 채점 함수

def normalize_text(t):
    return t.replace(',', '').replace(' ', '')

def normalize_dates(text):
    text = re.sub(r'(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일', r'\1.\2.\3', text)
    text = re.sub(r'(\d{4})\.(\d{1,2})\.(\d{1,2})', lambda m: f"{m.group(1)}.{int(m.group(2)):02d}.{int(m.group(3)):02d}", text)
    return text

def _text_included(fact_text, answer_text):
    fact_text = normalize_dates(fact_text)
    answer_text_n = normalize_dates(answer_text)
    answer_norm = normalize_text(answer_text_n)

    numbers = re.findall(r'\d+(?:\.\d+)?', fact_text)
    numbers = [n for n in numbers if len(n) >= 2]
    for num in numbers:
        if num in answer_norm:
            continue
        if len(num) == 4 and num.startswith('20'):
            if num[2:] in answer_norm:
                continue
        num_no_zero = re.sub(r'^0+', '', num)
        if num_no_zero and num_no_zero in answer_norm:
            continue
        return False

    raw_words = re.split(r'[\s,·:()]+', fact_text)
    stopwords_suffix = ('이다', '한다', '된다', '있다', '없다', '이며', '해야', '까지', '입니다', '부터', '년', '월', '일')
    core_words = []
    for w in raw_words:
        w = w.rstrip('.,')
        if len(w) < 2:
            continue
        if re.match(r'^\d', w):
            continue
        for suf in stopwords_suffix:
            if w.endswith(suf) and len(w) > len(suf):
                w = w[:-len(suf)]
                break
        w = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', w)
        if len(w) >= 2:
            core_words.append(w)

    if not core_words:
        return True

    match_count = sum(1 for w in core_words if w in answer_text or normalize_text(w) in answer_norm)
    return match_count / max(len(core_words), 1) >= 0.4

def _key_point_included(key_point, answer_text):
    if isinstance(key_point, dict):
        return _text_included(key_point['text'], answer_text)
    elif isinstance(key_point, list):
        return any(_text_included(variant, answer_text) for variant in key_point)
    else:
        return _text_included(str(key_point), answer_text)

ABSTAIN_PHRASES = ['확인되지 않습니다', '답변할 수 없', '수행할 수 없', '확인할 수 없', '판단할 수 없',
                    '판정할 수 없', '판정해줄 수 없', '계산할 수 없', '제공할 수 없', '받아들일 수 없',
                    '확정할 수 없', '알려드릴 수 없', '불가능합니다', '제공된 문서 범위에서는']

def official_score(item, answer_text):
    gold = item['gold']
    decision = gold.get('decision')

    if decision in ('abstain', 'source_conflict'):
        is_abstained = any(p in answer_text for p in ABSTAIN_PHRASES)
        return 100 if is_abstained else 0

    key_points = gold.get('required_key_points') or gold.get('required_fact_groups')
    if not key_points:
        return None

    included = [_key_point_included(kp, answer_text) for kp in key_points]
    return round(sum(included) / len(included) * 100, 2)

In [12]:
# rag-56 골든셋 답변 생성 + 채점

with open(DATA_DIR / 'rag-56.draft.jsonl', encoding='utf-8') as f:
    rag56 = [json.loads(l) for l in f if l.strip()]

rag56_answers = []
for item in rag56:
    cid = item['case_id']
    task_type = item['task_type']
    question = item['question']

    answer = ask_rfp_final(question)
    rag56_answers.append({'case_id': cid, 'task_type': task_type, 'answer': answer})
    print(f"[{cid}][{task_type}] {question}")
    print(answer)
    print()

[supplemental-qa-c01][single_doc] 평택시가 정류장 이용객에게 실시간 운행정보를 제공하려고 추진한 2024년 사업에는 예산이 얼마나 배정됐나요?
￦999,494,600원(부가세 포함)

근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp

[supplemental-qa-c02][single_doc] 평택시의 실시간 버스 도착정보 제공 체계를 새로 만드는 2024년 사업은 착수 후 언제까지 완료해야 하나요?
착수일로부터 2024.10.31. 완료해야 합니다. 근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp

[supplemental-qa-c03][single_doc] GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?
확인되지 않습니다. 제공된 문서들(한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp, 한국발명진흥회_2024년 건설기술…활용실.hwp, 울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp)에는 GKL의 2024년 정보화 사업 예산 관련 내용이 없습니다. 근거 문서: 한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp; 한국발명진흥회_2024년 건설기술에 관한 특허·실용신안 활용실.hwp; 울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp

[supplemental-qa-c04][single_doc] GKL이 사내 협업 업무 환경을 새로 만드는 사업은 어떤 경쟁 방식과 낙찰 절차로 계약하나요?
경쟁 방식: 제한경쟁입찰  
낙찰 절차(낙찰자 결정 방식): 협상에 의한 계약(기술평가 90% / 가격평가 10%) — 협상에 의한 계약으로 선정된 업체와 협상 후 계약 체결.  
근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp

[supplemental-qa-c05][single_d

In [13]:
# 채점

rag56_results = []
for a in rag56_answers:
    item = next(it for it in rag56 if it['case_id'] == a['case_id'])
    score = official_score(item, a['answer'])
    rag56_results.append({'case_id': a['case_id'], 'task_type': a['task_type'], 'score': score})
    print(f"[{a['case_id']}][{a['task_type']}] 점수: {score}")

valid_scores = [r['score'] for r in rag56_results if r['score'] is not None]
print(f"\n전체 평균: {sum(valid_scores)/len(valid_scores):.2f}/100 ({len(valid_scores)}개)")

by_type = {}
for r in rag56_results:
    if r['score'] is not None:
        by_type.setdefault(r['task_type'], []).append(r['score'])
for t, scores in by_type.items():
    print(f"{t}: 평균 {sum(scores)/len(scores):.2f}/100 ({len(scores)}개)")

[supplemental-qa-c01][single_doc] 점수: 100.0
[supplemental-qa-c02][single_doc] 점수: 100.0
[supplemental-qa-c03][single_doc] 점수: 0.0
[supplemental-qa-c04][single_doc] 점수: 100.0
[supplemental-qa-c05][single_doc] 점수: 100.0
[supplemental-qa-c06][single_doc] 점수: 100.0
[supplemental-qa-c07][single_doc] 점수: 100.0
[supplemental-qa-c08][single_doc] 점수: 100.0
[supplemental-qa-c09][single_doc] 점수: 0.0
[supplemental-qa-c10][single_doc] 점수: 100.0
[supplemental-qa-c11][single_doc] 점수: 100.0
[supplemental-qa-c12][single_doc] 점수: 33.33
[supplemental-qa-c13][single_doc] 점수: 100.0
[supplemental-qa-c14][single_doc] 점수: 100.0
[supplemental-qa-c15][single_doc] 점수: 100.0
[supplemental-qa-c16][single_doc] 점수: 50.0
[supplemental-qa-c18][single_doc] 점수: 0.0
[supplemental-qa-c19][multi_doc_compare] 점수: 66.67
[supplemental-qa-c20][multi_doc_compare] 점수: 100.0
[supplemental-qa-c23][single_doc] 점수: 0.0
[supplemental-qa-c25][single_doc] 점수: 0
[supplemental-qa-g01][single_doc] 점수: 100.0
[supplemental-qa-g02][single_do

In [14]:
zero_ids = ['supplemental-qa-c03','supplemental-qa-c09','supplemental-qa-c18','supplemental-qa-c23','supplemental-qa-g18','supplemental-qa-g23']
for a in rag56_answers:
    if a['case_id'] in zero_ids:
        print(f"[{a['case_id']}]")
        print(a['answer'])
        print()

[supplemental-qa-c03]
확인되지 않습니다. 제공된 문서들(한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp, 한국발명진흥회_2024년 건설기술…활용실.hwp, 울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp)에는 GKL의 2024년 정보화 사업 예산 관련 내용이 없습니다. 근거 문서: 한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp; 한국발명진흥회_2024년 건설기술에 관한 특허·실용신안 활용실.hwp; 울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp

[supplemental-qa-c09]
확인되지 않습니다. 근거 문서: 한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp

[supplemental-qa-c18]
확인되지 않습니다. 제공된 문서 범위에서는 나라장터(G2B) 등록 마감 기한 관련 내용이 확인되지 않습니다. 원문(제안요청서 전체)을 확인해 주세요. 근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp

[supplemental-qa-c23]
확인되지 않습니다. 제공된 문서 범위(한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp)에는 계약이행보증금 비율이 명시되어 있지 않습니다. 근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp

[supplemental-qa-g18]
확인되지 않습니다. 제공된 문서 조각에는 제안서 분량·기본 형식 규정의 구체 내용이 포함되어 있지 않습니다. 원문에서 아래 항목을 확인하세요.
- V. 제안일반사항 → 2. 제안서 작성지침 (문서 목차상 페이지 36)
- V. 제안일반사항 → 3. 제안서 제출 (문서 목차상 페이지 39)

근거: 부산관광공사_경영정보시스템 기능개선.hwp

[supplemental-qa-g23]
제공된 문서 범위에서는 제안서 본문 제한과 제안서 요약서 페이지 제한

In [15]:
matching = [f for f, biz in all_filenames_with_biz if 'GKL' in f or '그랜드코리아' in f or '카지노' in f]
print(matching)

['그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp']


In [16]:
ORG_ALIAS_MAP = {
    '대검찰청': ['검찰'],
    '고려대학교': ['고려대'],
    '한국산업단지공단': ['산단'],
    '그랜드코리아레저(주)': ['GKL'],
}

In [17]:
ORG_ALIAS_MAP['그랜드코리아레저(주)'] = ['GKL']

q = "GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?"
doc_hints = extract_doc_hints_multi(q, all_filenames_with_biz)
print(doc_hints)

['한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp', '한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp', '울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp', '서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용.pdf', '대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp', '한국연구재단_2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구.hwp', '국민연금공단_2024년 이러닝시스템 운영 용역.hwp', '(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp', '경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp', '국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축.hwp', '한국해양조사협회_2024년 항해용 간행물 품질관리 업무보조 시스템 구축.hwp', '재단법인 한국장애인문화예술원_2024년 장애인문화예술정보시스템 이음.hwp', '(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp', '경기도사회서비스원_2024년 통합사회정보시스템 운영지원.hwp', '문화체육관광부 국립민속박물관_2024년 국립민속박물관 민속아카이브 자.hwp', '조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp', '그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp', '재단법인 광주광역시 광주문화재단_2024년 광주문화예술통합플랫폼 시스.hwp']


In [18]:
q = "GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?"

stopwords_general = {'사업의', '사업에서', '사업은', '어떻게', '되나요', '되나요?', '몇', '어떤', '얼마', '비교', '알려줘', '정리해줘', '무엇인가요', '관련', '입찰공고일', '공고일', '입찰공고'}
raw_keywords = [w.rstrip('.,?!') for w in re.split(r'[ ,·]', q) if len(w) >= 4]
keywords_all = [w for w in raw_keywords if w not in stopwords_general and w not in COMMON_FILENAME_WORDS and '입찰공고' not in w]
print(keywords_all)

['GKL이', '전자결재', '구축하려는', '2024년', '얼마인가요']


In [19]:
COMMON_SUFFIX_WORDS.add('2024년')
COMMON_SUFFIX_WORDS.add('2025년')

q = "GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?"
doc_hints = extract_doc_hints_multi(q, all_filenames_with_biz)
print(doc_hints)

['한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp', '한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp', '울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp', '서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용.pdf', '대전대학교_대전대학교 2024학년도 다층적 융합 학습경험 플랫폼(MILE) 전.hwp', '한국연구재단_2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구.hwp', '국민연금공단_2024년 이러닝시스템 운영 용역.hwp', '(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp', '경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp', '국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축.hwp', '한국해양조사협회_2024년 항해용 간행물 품질관리 업무보조 시스템 구축.hwp', '재단법인 한국장애인문화예술원_2024년 장애인문화예술정보시스템 이음.hwp', '(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp', '경기도사회서비스원_2024년 통합사회정보시스템 운영지원.hwp', '문화체육관광부 국립민속박물관_2024년 국립민속박물관 민속아카이브 자.hwp', '조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp', '그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp', '재단법인 광주광역시 광주문화재단_2024년 광주문화예술통합플랫폼 시스.hwp']


In [20]:
q = "GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?"

print('2024년' in COMMON_FILENAME_WORDS)
print('2024년' in COMMON_SUFFIX_WORDS)

stopwords_general = {'사업의', '사업에서', '사업은', '어떻게', '되나요', '되나요?', '몇', '어떤', '얼마', '비교', '알려줘', '정리해줘', '무엇인가요', '관련', '입찰공고일', '공고일', '입찰공고'}
raw_keywords = [w.rstrip('.,?!') for w in re.split(r'[ ,·]', q) if len(w) >= 4]
keywords_all = [w for w in raw_keywords if w not in stopwords_general and w not in COMMON_FILENAME_WORDS and '입찰공고' not in w]
print(keywords_all)

False
True
['GKL이', '전자결재', '구축하려는', '2024년', '얼마인가요']


In [21]:
COMMON_FILENAME_WORDS.add('2024년')
COMMON_FILENAME_WORDS.add('2025년')

q = "GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?"
doc_hints = extract_doc_hints_multi(q, all_filenames_with_biz)
print(doc_hints)

[]


In [22]:
print(ORG_ALIAS_MAP)

org_part = '그랜드코리아레저(주)'
print(org_part in ORG_ALIAS_MAP)
print(ORG_ALIAS_MAP.get(org_part))

{'대검찰청': ['검찰'], '고려대학교': ['고려대'], '한국산업단지공단': ['산단'], '그랜드코리아레저(주)': ['GKL']}
True
['GKL']


In [23]:
fname = '그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp'
org_part = fname.replace('refined_', '').split('_')[0].strip()
org_core = re.sub(r'\s*\(.*?\)\s*', '', org_part).strip()
org_core_clean = re.sub(r'^\(사\)', '', org_core).strip()
print(f"org_part: '{org_part}'")
print(f"org_core: '{org_core}'")
print(f"org_core_clean: '{org_core_clean}'")
print(org_core_clean in ORG_ALIAS_MAP)

org_part: '그랜드코리아레저(주)'
org_core: '그랜드코리아레저'
org_core_clean: '그랜드코리아레저'
False


In [24]:
del ORG_ALIAS_MAP['그랜드코리아레저(주)']
ORG_ALIAS_MAP['그랜드코리아레저'] = ['GKL']

q = "GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?"
doc_hints = extract_doc_hints_multi(q, all_filenames_with_biz)
print(doc_hints)

['그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp']


In [25]:
answer = ask_rfp_final(q)
print(answer)

1,515,000천원 (부가세 포함). 근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp


In [26]:
# 기관명 별칭 / 공통 단어 블랙리스트 / 법률 키워드 맵 / 표 보정 데이터

ORG_ALIAS_MAP = {
    '대검찰청': ['검찰'],
    '고려대학교': ['고려대'],
    '한국산업단지공단': ['산단'],
    '그랜드코리아레저': ['GKL'],
}

COMMON_SUFFIX_WORDS = {
    '박물관', '시스템', '센터', '공단', '진흥원', '협회', '재단', '연구원', '공사', '대학교',
    '사업', '관리', '운영', '구축', '개선', '개발', '지원', '정보', '용역', '기관', '기술',
    '고도화', '확대', '기능', '서비스', '일자리', '플랫폼', '통합', '접수',
    '일자리재단', '일자리플랫폼', '보험', '입찰공고', '공고',
    '과학연구', '과학연', '학연구', '연구소', '기록관리', '경기기록',
    '학교', '학교 ', ' 학교', '산학협력단', '산학협력', '학협력단',
    '통합시스템',
    '2024년', '2025년',
}
COMMON_FILENAME_WORDS = COMMON_SUFFIX_WORDS | {'용역', '수립', '2차', '1차', '3차', '운영', '및', '구축용역'}

LEGAL_KEYWORDS_MAP = {
    '하도급': ['하도급'],
    '공동수급': ['공동수급', '지분율', '컨소시엄'],
    '지분율': ['지분율', '공동수급'],
    '계약보증금': ['계약보증금', '보증금'],
    '평가': ['배점', '평가비율', '기술평가', '가격평가'],
    '제안서 보상': ['제안서 보상'],
    '불이익': ['부정당업자', '입찰보증금', '귀속'],
    '제출물': ['제출서류', '부', 'USB', '제출규격'],
    '제출': ['제출서류', 'USB'],
    '수량': ['부', 'USB'],
    '구축기간': ['사업기간', '구축기간', '개월'],
    '사업기간': ['사업기간', '구축기간', '개월'],
    '유지보수': ['무상유지보수', '유지보수기간', '하자보수', '무상 하자보수'],
    '참가자격': ['참가자격', '참가 자격'],
    '유지관리': ['하자보수', '유지관리 인력', '무상 하자보수'],
    '교육 의무': ['유지관리 인력', '사용자 및 관리자', '하자보수'],
    '교육을': ['유지관리 인력', '사용자 및 관리자', '하자보수'],
    '검수 후': ['하자보수', '유지관리 인력'],
    '재입찰': ['재입찰', '재공고입찰', '최초의 입찰'],
    '재공고': ['재입찰', '재공고입찰', '최초의 입찰'],
    '조건 변경': ['재입찰', '재공고입찰', '최초의 입찰'],
    '지역 요건': ['주된 영업소', '소재지'],
    '부산에': ['주된 영업소', '소재지'],
    '지역요건': ['주된 영업소', '소재지'],
    '소재지': ['주된 영업소', '소재지'],
    '보유인력': ['보유인력', '배점한도'],
    '배점한도': ['보유인력', '배점한도'],
    '계량평가': ['보유인력', '배점한도', '재무구조'],
    '규모비율': ['규모비율', '환산점수', '점수비중'],
    '환산점수': ['규모비율', '환산점수', '점수비중'],
    '수행실적': ['규모비율', '환산점수', '수행실적'],
    '신인도': ['신인도', '가점'],
    '가점표': ['신인도', '가점'],
    '연구원 승인': ['Lesson', '회람'],
    '발생한 경우': ['Lesson', '회람'],
    '회람': ['Lesson', '회람'],
}

In [27]:
q = "한국철도공사가 열차 운행기록을 자동으로 분석하는 체계를 개선하는 용역은 착수 후 며칠 동안 진행되나요?"
doc_hints = extract_doc_hints_multi(q, all_filenames_with_biz)
print(doc_hints)

matching = [f for f, biz in all_filenames_with_biz if '한국철도공사' in f]
print(matching)

['한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp']
['한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp', '한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp', '한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp']


In [28]:
q = "한국철도공사가 열차 운행기록을 자동으로 분석하는 체계를 개선하는 용역은 착수 후 며칠 동안 진행되나요?"
stopwords = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교'}
keywords = [w for w in re.split(r'[ ,]', q) if len(w) >= 2 and w not in stopwords]
print(keywords)

for fname in ['한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp',
              '한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp',
              '한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp']:
    score2 = sum(1 for kw in keywords if kw in fname)
    print(f"{fname[:40]} | score={score2}")

['한국철도공사가', '열차', '운행기록을', '자동으로', '분석하는', '체계를', '개선하는', '용역은', '착수', '며칠', '동안', '진행되나요?']
한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp | score=0
한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).h | score=0
한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스. | score=0


In [29]:
def _debug_fuzzy(kw, text, min_overlap=3):
    kw_ns = kw.replace(' ', '')
    text_ns = text.replace(' ', '')
    if kw_ns in text_ns:
        return True
    for n in range(len(kw_ns), min_overlap - 1, -1):
        if kw_ns[:n] in text_ns:
            return True
    return False

q = "한국철도공사가 열차 운행기록을 자동으로 분석하는 체계를 개선하는 용역은 착수 후 며칠 동안 진행되나요?"
stopwords = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교'}
keywords = [w for w in re.split(r'[ ,]', q) if len(w) >= 2 and w not in stopwords]

for fname in ['한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp',
              '한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp',
              '한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp']:
    score2 = sum(1 for kw in keywords if _debug_fuzzy(kw, fname))
    print(f"{fname[:40]} | score={score2}")

한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp | score=1
한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).h | score=1
한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스. | score=1


In [30]:
for fname in ['한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp',
              '한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp',
              '한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp']:
    matched = [kw for kw in keywords if _debug_fuzzy(kw, fname)]
    print(f"{fname[:40]} | matched={matched}")

한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp | matched=['한국철도공사가']
한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).h | matched=['한국철도공사가']
한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스. | matched=['한국철도공사가']


In [31]:
def _debug_fuzzy(kw, text, min_overlap=4):
    kw_ns = kw.replace(' ', '')
    text_ns = text.replace(' ', '')
    if kw_ns in text_ns:
        return True
    for n in range(len(kw_ns), min_overlap - 1, -1):
        if kw_ns[:n] in text_ns:
            return True
    return False

q = "한국철도공사가 열차 운행기록을 자동으로 분석하는 체계를 개선하는 용역은 착수 후 며칠 동안 진행되나요?"
stopwords = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교', '한국철도공사가', '한국철도공사'}
keywords = [w for w in re.split(r'[ ,]', q) if len(w) >= 2 and w not in stopwords]
print(keywords)

for fname in ['한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp',
              '한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp',
              '한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp']:
    matched = [kw for kw in keywords if _debug_fuzzy(kw, fname)]
    print(f"{fname[:40]} | matched={matched}")

['열차', '운행기록을', '자동으로', '분석하는', '체계를', '개선하는', '용역은', '착수', '며칠', '동안', '진행되나요?']
한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp | matched=[]
한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).h | matched=[]
한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스. | matched=[]


In [32]:
def _debug_fuzzy2(kw, text, min_chunk=2):
    kw_ns = kw.replace(' ', '')
    text_ns = text.replace(' ', '')

    for i in range(len(kw_ns) - min_chunk + 1):
        chunk = kw_ns[i:i+min_chunk+1] if i+min_chunk+1 <= len(kw_ns) else kw_ns[i:]
        if len(chunk) >= min_chunk and chunk in text_ns:
            return True
    return False

q = "한국철도공사가 열차 운행기록을 자동으로 분석하는 체계를 개선하는 용역은 착수 후 며칠 동안 진행되나요?"
stopwords = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교', '한국철도공사가', '한국철도공사'}
keywords = [w for w in re.split(r'[ ,]', q) if len(w) >= 2 and w not in stopwords]

for fname in ['한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp',
              '한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp',
              '한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp']:
    matched = [kw for kw in keywords if _debug_fuzzy2(kw, fname)]
    print(f"{fname[:40]} | matched={matched}")

한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp | matched=[]
한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).h | matched=[]
한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스. | matched=[]


In [33]:
def _debug_fuzzy3(kw, text, min_len=2):
    kw_clean = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', kw)
    if len(kw_clean) < min_len:
        return False
    return kw_clean in text.replace(' ', '')

q = "한국철도공사가 열차 운행기록을 자동으로 분석하는 체계를 개선하는 용역은 착수 후 며칠 동안 진행되나요?"
stopwords = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교', '한국철도공사가', '한국철도공사'}
keywords = [w for w in re.split(r'[ ,]', q) if len(w) >= 2 and w not in stopwords]

for fname in ['한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp',
              '한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp',
              '한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp']:
    matched = [kw for kw in keywords if _debug_fuzzy3(kw, fname)]
    print(f"{fname[:40]} | matched={matched}")

한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp | matched=['용역은']
한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).h | matched=['용역은']
한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스. | matched=['자동으로', '용역은']


In [34]:
# 기관명 별칭 / 공통 단어 블랙리스트 / 법률 키워드 맵

ORG_ALIAS_MAP = {
    '대검찰청': ['검찰'],
    '고려대학교': ['고려대'],
    '한국산업단지공단': ['산단'],
    '그랜드코리아레저': ['GKL'],
}

COMMON_SUFFIX_WORDS = {
    '박물관', '시스템', '센터', '공단', '진흥원', '협회', '재단', '연구원', '공사', '대학교',
    '사업', '관리', '운영', '구축', '개선', '개발', '지원', '정보', '용역', '기관', '기술',
    '고도화', '확대', '기능', '서비스', '일자리', '플랫폼', '통합', '접수',
    '일자리재단', '일자리플랫폼', '보험', '입찰공고', '공고',
    '과학연구', '과학연', '학연구', '연구소', '기록관리', '경기기록',
    '학교', '학교 ', ' 학교', '산학협력단', '산학협력', '학협력단',
    '통합시스템',
    '2024년', '2025년',
}
COMMON_FILENAME_WORDS = COMMON_SUFFIX_WORDS | {'용역', '수립', '2차', '1차', '3차', '운영', '및', '구축용역'}

LEGAL_KEYWORDS_MAP = {
    '하도급': ['하도급'],
    '공동수급': ['공동수급', '지분율', '컨소시엄'],
    '지분율': ['지분율', '공동수급'],
    '계약보증금': ['계약보증금', '보증금'],
    '평가': ['배점', '평가비율', '기술평가', '가격평가'],
    '제안서 보상': ['제안서 보상'],
    '불이익': ['부정당업자', '입찰보증금', '귀속'],
    '제출물': ['제출서류', '부', 'USB', '제출규격'],
    '제출': ['제출서류', 'USB'],
    '수량': ['부', 'USB'],
    '구축기간': ['사업기간', '구축기간', '개월'],
    '사업기간': ['사업기간', '구축기간', '개월'],
    '유지보수': ['무상유지보수', '유지보수기간', '하자보수', '무상 하자보수'],
    '참가자격': ['참가자격', '참가 자격'],
    '유지관리': ['하자보수', '유지관리 인력', '무상 하자보수'],
    '교육 의무': ['유지관리 인력', '사용자 및 관리자', '하자보수'],
    '교육을': ['유지관리 인력', '사용자 및 관리자', '하자보수'],
    '검수 후': ['하자보수', '유지관리 인력'],
    '재입찰': ['재입찰', '재공고입찰', '최초의 입찰'],
    '재공고': ['재입찰', '재공고입찰', '최초의 입찰'],
    '조건 변경': ['재입찰', '재공고입찰', '최초의 입찰'],
    '지역 요건': ['주된 영업소', '소재지'],
    '부산에': ['주된 영업소', '소재지'],
    '지역요건': ['주된 영업소', '소재지'],
    '소재지': ['주된 영업소', '소재지'],
    '보유인력': ['보유인력', '배점한도'],
    '배점한도': ['보유인력', '배점한도'],
    '계량평가': ['보유인력', '배점한도', '재무구조'],
    '규모비율': ['규모비율', '환산점수', '점수비중'],
    '환산점수': ['규모비율', '환산점수', '점수비중'],
    '수행실적': ['규모비율', '환산점수', '수행실적'],
    '신인도': ['신인도', '가점'],
    '가점표': ['신인도', '가점'],
    '연구원 승인': ['Lesson', '회람'],
    '발생한 경우': ['Lesson', '회람'],
    '회람': ['Lesson', '회람'],
}

In [35]:
# 조건 필터링 + 검색 함수

def find_relevant_keywords(question):
    matched = []
    for trigger, kws in LEGAL_KEYWORDS_MAP.items():
        if trigger in question:
            matched.extend(kws)
    return list(set(matched))

def is_aggregation_question(question):
    keywords = ['몇 개', '개수', '다 나열', '몇 건']
    strong_total = '전부' in question or ('총' in question and ('개' in question or '건' in question))
    return any(kw in question for kw in keywords) or strong_total

def extract_filter_conditions(query):
    conditions = {}
    if '억' in query and ('이상' in query or '넘는' in query):
        match = re.search(r'(\d+)억', query)
        if match:
            conditions['금액_최소'] = int(match.group(1)) * 100000000
    if '지자체' in query or '지방자치단체' in query:
        conditions['지자체'] = True
    if '공사' in query and ('OO공사' in query or '발주기관이' in query):
        conditions['공사'] = True
    if 'AI' in query:
        conditions['주제_AI'] = True
    if '긴급' in query:
        conditions['긴급'] = True
    if '보안' in query:
        conditions['보안'] = True
    if '재난' in query:
        conditions['재난'] = True
    return conditions

def is_local_gov(org):
    if org is None or (isinstance(org, float)):
        return False
    return bool(re.search(r'(광역시|특별시|특별자치도|특별자치시|[가-힣]+도|[가-힣]+시|[가-힣]+군|[가-힣]+구)$', str(org).strip()))

def get_filtered_candidates(conditions, chunk_metadata):
    if not conditions:
        return None
    doc_info = {}
    for cm in chunk_metadata:
        fname = cm['파일명']
        if fname not in doc_info:
            doc_info[fname] = cm
    allowed = set()
    for fname, info in doc_info.items():
        ok = True
        if '금액_최소' in conditions:
            amt = info.get('사업금액')
            if amt is None or amt < conditions['금액_최소']:
                ok = False
        if conditions.get('지자체'):
            if not is_local_gov(info.get('발주기관')):
                ok = False
        if conditions.get('공사'):
            org = str(info.get('발주기관', ''))
            if '공사' not in org:
                ok = False
        if conditions.get('긴급'):
            if '긴급' not in fname:
                ok = False
        if conditions.get('재난'):
            if '재난' not in fname:
                ok = False
        if ok:
            allowed.add(fname)
    return allowed if allowed else None

def search_with_filter(query, index, model, chunk_metadata, all_chunks, k=10, max_per_doc=1):
    conditions = extract_filter_conditions(query)
    allowed_filenames = get_filtered_candidates(conditions, chunk_metadata)
    query_embedding = model.encode([query])
    search_k = min(len(all_chunks), 2000)
    distances, indices = index.search(np.array(query_embedding).astype('float32'), search_k)
    seen_docs = {}
    results = []
    for i in indices[0]:
        doc_name = chunk_metadata[i]['파일명']
        if allowed_filenames is not None and doc_name not in allowed_filenames:
            continue
        count = seen_docs.get(doc_name, 0)
        if count < max_per_doc:
            results.append(i)
            seen_docs[doc_name] = count + 1
        if len(results) >= k:
            break
    return results

def normalize_org_name(name):
    return re.sub(r'(특별시|광역시|특별자치시|특별자치도)', '', name)

In [36]:
# 문서 힌트 추출

def extract_doc_hints_multi(question, all_filenames_with_biz):
    q_no_space = question.replace(' ', '').replace('&', '')
    org_candidates = []
    for fname, biz_name in all_filenames_with_biz:
        org_part = fname.replace('refined_', '').split('_')[0].strip()
        org_core = re.sub(r'\s*\(.*?\)\s*', '', org_part).strip()
        org_core_clean = re.sub(r'^\(사\)', '', org_core).strip()
        org_core_clean = re.sub(r'\s*입찰공고\s*$', '', org_core_clean).strip()
        org_core_norm = normalize_org_name(org_core_clean)
        if len(org_core_clean) < 2:
            continue

        matched = False
        if org_core_clean in question:
            matched = True
        elif len(org_core_norm) >= 3 and org_core_norm in question:
            matched = True
        elif org_core_clean in ORG_ALIAS_MAP and any(alias in question for alias in ORG_ALIAS_MAP[org_core_clean]):
            matched = True
        else:
            min_len = 4
            for target_str in [org_core_clean, org_core_norm]:
                for start in range(len(target_str) - min_len + 1):
                    for length in range(len(target_str) - start, min_len - 1, -1):
                        substr = target_str[start:start+length]
                        if substr.strip() in question and substr.strip() not in COMMON_SUFFIX_WORDS:
                            matched = True
                            break
                    if matched:
                        break
                if matched:
                    break
        if matched:
            org_candidates.append((fname, org_core_clean))

    biz_candidates = []
    quoted = re.findall(r"['\"]([^'\"]+)['\"]", question)
    for fname, biz_name in all_filenames_with_biz:
        biz_name = str(biz_name).strip()
        if len(biz_name) >= 4 and biz_name in question:
            biz_candidates.append(fname)
            continue
        for q in quoted:
            if q in biz_name or biz_name in q:
                biz_candidates.append(fname)
                break
        eng_words = re.findall(r'[A-Za-z][A-Za-z&\s]{2,}[A-Za-z]', biz_name)
        for ew in eng_words:
            ew_no_space = ew.strip().replace(' ', '').replace('&', '')
            if len(ew_no_space) >= 4 and ew_no_space in q_no_space:
                biz_candidates.append(fname)
                break

    stopwords_general = {'사업의', '사업에서', '사업은', '어떻게', '되나요', '되나요?', '몇', '어떤', '얼마', '비교', '알려줘', '정리해줘', '무엇인가요', '관련', '입찰공고일', '공고일', '입찰공고'}
    raw_keywords = [w.rstrip('.,?!') for w in re.split(r'[ ,·]', question) if len(w) >= 4]
    keywords_all = [w for w in raw_keywords if w not in stopwords_general and w not in COMMON_FILENAME_WORDS and '입찰공고' not in w]

    def fuzzy_match(kw, text, min_overlap=4):
        kw_ns = kw.replace(' ', '')
        text_ns = text.replace(' ', '')
        if kw_ns in text_ns:
            return True
        for n in range(len(kw_ns), min_overlap - 1, -1):
            if kw_ns[:n] in text_ns:
                return True
        return False

    def keyword_weight(kw):
        return 3 if re.search(r'[A-Za-z]', kw) else 1

    filename_candidates = []
    for fname, biz_name in all_filenames_with_biz:
        fname_clean = fname.replace('refined_', '').replace('.hwp', '').replace('.pdf', '')
        matched_kws = [kw for kw in keywords_all if fuzzy_match(kw, fname_clean)]
        score = sum(keyword_weight(kw) for kw in matched_kws)
        if score > 0:
            filename_candidates.append((fname, score, len(matched_kws)))

    if filename_candidates:
        filename_candidates.sort(key=lambda x: -x[1])
        max_score = filename_candidates[0][1]
        for top_fname, score, cnt in filename_candidates:
            if score >= max_score * 0.6 or score >= 1:
                if top_fname not in [f for f, _ in org_candidates] and top_fname not in biz_candidates:
                    if len(filename_candidates) <= 3 or score >= max(max_score * 0.6, 1):
                        biz_candidates.append(top_fname)

    org_groups = {}
    for fname, org_core in org_candidates:
        org_groups.setdefault(org_core, []).append(fname)

    stopwords = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교'}
    keywords = [w for w in re.split(r'[ ,]', question) if len(w) >= 2 and w not in stopwords]

    final_hints = []
    for org_core, fnames in org_groups.items():
        fnames = list(set(fnames))
        if len(fnames) == 1:
            final_hints.append(fnames[0])
        else:
            fname_to_biz = dict(all_filenames_with_biz)
            best_doc, best_score2 = None, -1
            for fname in fnames:
                biz_name = fname_to_biz.get(fname, '')
                score2 = sum(1 for kw in keywords if kw in fname or kw in str(biz_name))
                if score2 > best_score2:
                    best_score2, best_doc = score2, fname
            final_hints.append(best_doc)

    for fname in biz_candidates:
        if fname not in final_hints:
            final_hints.append(fname)

    return list(dict.fromkeys(final_hints))

In [37]:
rag56_answers = []
for item in rag56:
    cid = item['case_id']
    task_type = item['task_type']
    question = item['question']

    answer = ask_rfp_final(question)
    rag56_answers.append({'case_id': cid, 'task_type': task_type, 'answer': answer})
    print(f"[{cid}][{task_type}] {question}")
    print(answer)
    print()

[supplemental-qa-c01][single_doc] 평택시가 정류장 이용객에게 실시간 운행정보를 제공하려고 추진한 2024년 사업에는 예산이 얼마나 배정됐나요?
￦999,494,600원(부가세 포함). 근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp

[supplemental-qa-c02][single_doc] 평택시의 실시간 버스 도착정보 제공 체계를 새로 만드는 2024년 사업은 착수 후 언제까지 완료해야 하나요?
착수일로부터 2024. 10. 31.까지 완료해야 합니다. 근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp

[supplemental-qa-c03][single_doc] GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?
사업예산: 1,515,000천원 (부가세 포함). 근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp

[supplemental-qa-c04][single_doc] GKL이 사내 협업 업무 환경을 새로 만드는 사업은 어떤 경쟁 방식과 낙찰 절차로 계약하나요?
계약방법: 제한경쟁입찰(협상에 의한 계약)  
낙찰(평가)절차: 기술평가 90% + 가격평가 10%로 종합평가 후 낙찰자 선정

근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp

[supplemental-qa-c05][single_doc] 한국농어촌공사가 네팔에서 물 관리 정보를 디지털화하기 위해 추진하는 시범 시스템은 착수 후 얼마 동안 수행하나요?
용역착수일로부터 6개월 수행합니다. 근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp

[supplemental-qa-c06][single_doc] 한국농어촌공사가 네팔의 수자원 정보를 관리할 시범 시스템을 만드는 데 편성한 예산은 얼마인가요? 세금 포함 여부도 알려주세요.
금 181,913,000원 

In [38]:
# 재채점

rag56_results = []
for a in rag56_answers:
    item = next(it for it in rag56 if it['case_id'] == a['case_id'])
    score = official_score(item, a['answer'])
    rag56_results.append({'case_id': a['case_id'], 'task_type': a['task_type'], 'score': score})
    print(f"[{a['case_id']}][{a['task_type']}] 점수: {score}")

valid_scores = [r['score'] for r in rag56_results if r['score'] is not None]
print(f"\n전체 평균: {sum(valid_scores)/len(valid_scores):.2f}/100 ({len(valid_scores)}개)")

by_type = {}
for r in rag56_results:
    if r['score'] is not None:
        by_type.setdefault(r['task_type'], []).append(r['score'])
for t, scores in by_type.items():
    print(f"{t}: 평균 {sum(scores)/len(scores):.2f}/100 ({len(scores)}개)")

[supplemental-qa-c01][single_doc] 점수: 100.0
[supplemental-qa-c02][single_doc] 점수: 100.0
[supplemental-qa-c03][single_doc] 점수: 100.0
[supplemental-qa-c04][single_doc] 점수: 100.0
[supplemental-qa-c05][single_doc] 점수: 100.0
[supplemental-qa-c06][single_doc] 점수: 100.0
[supplemental-qa-c07][single_doc] 점수: 100.0
[supplemental-qa-c08][single_doc] 점수: 100.0
[supplemental-qa-c09][single_doc] 점수: 0.0
[supplemental-qa-c10][single_doc] 점수: 100.0
[supplemental-qa-c11][single_doc] 점수: 0.0
[supplemental-qa-c12][single_doc] 점수: 66.67
[supplemental-qa-c13][single_doc] 점수: 100.0
[supplemental-qa-c14][single_doc] 점수: 100.0
[supplemental-qa-c15][single_doc] 점수: 100.0
[supplemental-qa-c16][single_doc] 점수: 50.0
[supplemental-qa-c18][single_doc] 점수: 0.0
[supplemental-qa-c19][multi_doc_compare] 점수: 66.67
[supplemental-qa-c20][multi_doc_compare] 점수: 100.0
[supplemental-qa-c23][single_doc] 점수: 0.0
[supplemental-qa-c25][single_doc] 점수: 0
[supplemental-qa-g01][single_doc] 점수: 100.0
[supplemental-qa-g02][single_do

In [39]:
check_ids = ['supplemental-qa-c11', 'supplemental-qa-g08', 'supplemental-qa-g11', 'supplemental-qa-g21']
for a in rag56_answers:
    if a['case_id'] in check_ids:
        print(f"[{a['case_id']}]")
        print(a['answer'])
        print()

[supplemental-qa-c11]
제공된 문서 범위에서는 확인되지 않습니다. 근거 문서: 한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp

[supplemental-qa-g08]
제공된 문서 범위에서는 제안 관련 제출물의 수량이 확인되지 않습니다. 원문 전체(제출서류·제안서 제출일정 섹션) 확인이 필요할 수 있습니다.

근거: 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시ystems 고도화 제안요청서 (문서)

[supplemental-qa-g11]
제공된 문서 범위에서는 판단할 수 없습니다. 입찰참가자격 세부조건(법적 자격요건, 자본금·사업자 유형 제한, 지역요건 등)은 문서의 'Ⅵ.사업자 선정 → 3. 입찰참가자격(페이지 41)'에 명시되어 있으니 해당 항목 원문을 확인해 주세요.

근거: 부산관광공사_경영정보시스템 기능개선.hwp

[supplemental-qa-g21]
제공된 문서 범위에서는 확인되지 않습니다. 원문의 'Ⅴ. 입찰 및 제안 안내 → 1. 입찰 참가 자격' 항목을 확인해야 합니다. 근거 문서: 재단법인경기도일자리재단_2025년 통합접수시스템 운영.hwp

